# Microcosmic God → Catch Transfer Test

This notebook tests whether brains evolved in the [microcosmic-god](https://github.com/asystemoffields/microcosmic-god) artificial-life sandbox have transferable representations — does the cognition that emerged in a complex causal-survival world transfer to a simple paddle-and-ball game it has never seen?

## The protocol

1. Load a brain from a microcosmic-god checkpoint. **Brain weights stay frozen — no further learning.**
2. Train a small linear adapter that maps Catch's 4-dim observation into the brain's 72-dim input space, and the brain's 15-dim output into Catch's 3 actions.
3. Compare three conditions:
   - **Random policy** — left/stay/right uniformly. Lower bound.
   - **Untrained brain + trained adapter** — brain weights random-init; only adapter learns. Controls for what the adapter alone can do.
   - **Trained brain (loaded checkpoint) + trained adapter** — the actual transfer test.

If condition 3 outperforms condition 2 meaningfully, the brain's internal representations are doing useful work in a task they were never trained on. That's a real signal of transferability.

**Why a linear adapter and not a deeper one?** Deep adapters can learn to compute Catch from scratch, hiding any (lack of) contribution from the brain. A small linear projection is the cleanest test: the brain has to be doing something useful for the adapter to extract a working policy.

## Setup

Pure numpy + matplotlib. Clones the microcosmic-god repo to access sample checkpoints.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('microcosmic-god'):
    !git clone -q https://github.com/asystemoffields/microcosmic-god.git

BRAIN_REPO = 'microcosmic-god' if os.path.exists('microcosmic-god') else '.'
print('Using brain repo at:', BRAIN_REPO)

## The Catch environment

10×10 grid. Paddle on bottom row, 3 cells wide. Ball drops one row per tick from a random column. Three actions: left / stay / right. Reward +1 if caught, −1 if missed.

**Multi-ball mode (default):** an "episode" is `balls_per_episode` consecutive balls — paddle persists across balls, only the ball respawns. With `balls_per_episode=50` an episode is ~500 ticks, matching the brain's native operating horizon (mg organisms live 100–1000s of ticks). Set `balls_per_episode=1` for the single-ball mode used in earlier experiments.

Observation: `[paddle_x, ball_x, ball_y, t]` all normalized to [0, 1] — 4 floats.

In [ ]:
class Catch:
    def __init__(self, size=10, paddle_width=3, balls_per_episode=50, rng=None):
        self.size = size
        self.paddle_width = paddle_width
        self.balls_per_episode = balls_per_episode
        self.rng = rng if rng is not None else np.random.default_rng()
        self.reset()

    def _spawn_ball(self):
        self.ball_x = int(self.rng.integers(0, self.size))
        self.ball_y = 0

    def reset(self):
        self.paddle_x = self.size // 2
        self._spawn_ball()
        self.t = 0
        self.balls_completed = 0
        return self._obs()

    def _obs(self):
        return np.array([
            self.paddle_x / max(1, self.size - 1),
            self.ball_x / max(1, self.size - 1),
            self.ball_y / max(1, self.size - 1),
            (self.t % self.size) / max(1, self.size - 1),  # within-ball tick, normalized
        ], dtype=np.float64)

    def step(self, action: int):
        if action == 0:
            self.paddle_x = max(0, self.paddle_x - 1)
        elif action == 2:
            self.paddle_x = min(self.size - 1, self.paddle_x + 1)
        self.ball_y += 1
        self.t += 1
        ball_landed = self.ball_y >= self.size - 1
        reward = 0.0
        if ball_landed:
            half = self.paddle_width // 2
            reward = 1.0 if abs(self.ball_x - self.paddle_x) <= half else -1.0
            self.balls_completed += 1
            if self.balls_completed >= self.balls_per_episode:
                # Episode terminates after balls_per_episode balls.
                return self._obs(), reward, True
            # Spawn next ball; paddle persists (multi-ball semantics).
            self._spawn_ball()
        return self._obs(), reward, False

## Brain inference

Loads a microcosmic-god `TinyBrain` checkpoint. Pure numpy, forward-pass only — no plasticity by default. Mirrors the math in `microcosmic_god/brain.py`'s `forward()`, `_attend()`, `_retrieve_episodes()`, and `replay_episode()`. Attention noise is set to zero for deterministic evaluation.

Supports v2 brains with episodic memory: if the checkpoint contains `episodic_slots` and `episodic_age`, the inference class loads them, blends retrieved episodes into the recurrent dynamics each forward pass, and exposes `replay_episode()` so the adapter can trigger consolidation when the policy chooses Catch's "stay" action (the closest analog to mg's `rest`).

In [ ]:
import random as _python_random


class TinyBrainInference:
    """v2-capable inference. Forward + (optional) plasticity for adaptive
    transfer. Loads episodic memory from checkpoint if present."""

    def __init__(self, weights_in, weights_out, bias_h, bias_o,
                 attention_weights=None, attention_bias=None,
                 episodic_slots=None, episodic_age=None):
        self.weights_in = weights_in
        self.weights_out = weights_out
        self.bias_h = bias_h
        self.bias_o = bias_o
        self.attention_weights = attention_weights if (attention_weights is not None and attention_weights.size) else None
        self.attention_bias = attention_bias if (attention_bias is not None and attention_bias.size) else None
        self.episodic_slots = episodic_slots if (episodic_slots is not None and episodic_slots.size) else None
        self.episodic_age = episodic_age if (episodic_age is not None and episodic_age.size) else None
        self.input_size = weights_in.shape[1]
        self.hidden_size = weights_in.shape[0]
        self.output_size = weights_out.shape[0]
        self.hidden = np.zeros(self.hidden_size)
        self.last_inputs = np.zeros(self.input_size)

    @classmethod
    def from_checkpoint(cls, path):
        with open(path) as f:
            data = json.load(f)
        bd = data.get('brain') or data.get('brain_template') or data
        h = int(bd['hidden_size'])
        i = int(bd['input_size'])
        o = int(bd['output_size'])
        weights_in = np.array(bd['weights_in'], dtype=np.float64).reshape(h, i)
        weights_out = np.array(bd['weights_out'], dtype=np.float64).reshape(o, h)
        bias_h = np.array(bd['bias_h'], dtype=np.float64)
        bias_o = np.array(bd['bias_o'], dtype=np.float64)
        aw = bd.get('attention_weights') or []
        ab = bd.get('attention_bias') or []
        attention_weights = np.array(aw, dtype=np.float64).reshape(h, i) if len(aw) == h * i else None
        attention_bias = np.array(ab, dtype=np.float64) if len(ab) == i else None
        cap = int(bd.get('episodic_capacity') or 0)
        es_raw = bd.get('episodic_slots') or []
        ea_raw = bd.get('episodic_age') or []
        episodic_slots = np.array(es_raw, dtype=np.float64).reshape(cap, h) if cap > 0 and len(es_raw) == cap * h else None
        episodic_age = np.array(ea_raw, dtype=np.float64) if cap > 0 and len(ea_raw) == cap else None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias, episodic_slots, episodic_age)

    @classmethod
    def random(cls, input_size, hidden_size, output_size, rng=None,
               with_attention=True, with_episodic=False, episodic_capacity=0):
        rng = rng or np.random.default_rng(0)
        weights_in = rng.normal(0, 0.45, (hidden_size, input_size))
        weights_out = rng.normal(0, 0.45, (output_size, hidden_size))
        bias_h = rng.normal(0, 0.08, hidden_size)
        bias_o = rng.normal(0, 0.08, output_size)
        attention_weights = rng.normal(0, 0.15, (hidden_size, input_size)) if with_attention else None
        attention_bias = rng.normal(3.0, 0.10, input_size) if with_attention else None
        episodic_slots = np.zeros((episodic_capacity, hidden_size)) if (with_episodic and episodic_capacity > 0) else None
        episodic_age = np.full(episodic_capacity, -1.0) if (with_episodic and episodic_capacity > 0) else None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias, episodic_slots, episodic_age)

    def reset(self):
        # Zero only the recurrent state. Episodic slots persist (frozen since
        # training; resetting them would erase the brain's memories).
        self.hidden = np.zeros(self.hidden_size)
        self.last_inputs = np.zeros(self.input_size)

    def _has_episodic(self):
        return self.episodic_slots is not None

    def _attend(self, x, budget_fraction=0.95):
        if self.attention_weights is None:
            return x
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        logits = self.attention_bias + (self.hidden @ self.attention_weights) * inv_h
        np.clip(logits, -30.0, 30.0, out=logits)
        raw = 1.0 / (1.0 + np.exp(-logits))
        budget = self.input_size * budget_fraction
        total = raw.sum()
        fidelity = raw * (budget / total) if total > budget and total > 0 else raw
        return x * fidelity

    def _retrieve_episodes(self):
        if not self._has_episodic():
            return np.zeros(self.hidden_size)
        norm = np.sqrt(max(1.0, float(self.hidden_size)))
        scores = (self.episodic_slots @ self.hidden) / norm
        scores -= scores.max()
        weights = np.exp(scores)
        weights /= weights.sum() + 1e-9
        return weights @ self.episodic_slots

    def replay_episode(self, rng=None):
        if not self._has_episodic() or self.episodic_slots.shape[0] < 2:
            return False
        valid_indices = np.flatnonzero(self.episodic_age >= 0.0)
        if valid_indices.size < 2:
            return False
        rng = rng or _python_random
        i_a, i_b = rng.sample(valid_indices.tolist(), 2)
        replay_state = 0.5 * (self.episodic_slots[i_a] + self.episodic_slots[i_b])
        self.hidden = np.tanh(self.bias_h + 0.62 * replay_state)
        return True

    def forward(self, inputs):
        x = self._attend(np.asarray(inputs, dtype=np.float64))
        self.last_inputs = x
        inv = 1.0 / np.sqrt(max(1, self.input_size))
        new_hidden = np.tanh(self.bias_h + 0.62 * self.hidden + (self.weights_in @ x) * inv)
        if self._has_episodic():
            self.hidden = new_hidden
            retrieved = self._retrieve_episodes()
            new_hidden = 0.82 * new_hidden + 0.18 * retrieved
        self.hidden = new_hidden
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        return self.bias_o + (self.weights_out @ self.hidden) * inv_h

    def learn(self, action_index, valence, learning_rate=0.10, plasticity=0.5):
        """Slimmed plasticity for adaptive-transfer mode. Reward-modulated
        Hebbian on action policy + representation. No prediction-head learning
        here (those need targets we don't synthesize for Catch). Mirrors the
        relevant subset of microcosmic_god.brain.TinyBrain.learn()."""
        if action_index < 0 or action_index >= self.output_size:
            return
        valence = max(-2.0, min(2.0, valence))
        lr = max(0.0, min(0.25, learning_rate)) * max(0.0, min(1.0, plasticity))
        # Action policy update: nudge weights_out for the chosen action toward hidden.
        self.weights_out[action_index] = np.clip(
            self.weights_out[action_index] + lr * valence * 0.035 * self.hidden,
            -4.0, 4.0,
        )
        self.bias_o[action_index] = max(
            -4.0, min(4.0, self.bias_o[action_index] + lr * valence * 0.015)
        )
        # Representation update: Hebbian-ish on weights_in.
        repr_lr = lr * 0.020
        if repr_lr > 0.0 and self.last_inputs.size:
            hidden_gate = np.clip(self.hidden, -1.0, 1.0)
            delta = repr_lr * valence * np.outer(hidden_gate, self.last_inputs)
            self.weights_in = np.clip(self.weights_in + delta, -4.0, 4.0)

## The adapter

Two trainable linear projections wrapped around the (frozen) brain:
- `W_in` — Catch's 4-dim observation → brain's input space (72 dims by default)
- `W_out` — brain's 15-dim output → 3 Catch actions

The brain itself is unchanged. If the brain's hidden representations are useful for Catch, training only these projections should produce a competent player.

**Architectural alignment with the brain (added):**
- Hidden state **persists across balls within an episode** (multi-ball mode). The adapter's `reset()` is called per-episode, not per-ball; this matches the brain's native horizon.
- When the policy chooses Catch's **stay** action (the closest analog to mg's `rest`), the adapter calls `brain.replay_episode()` so the v2 episodic-replay/consolidation machinery actually fires during the test.

In [ ]:
_REPLAY_RNG = _python_random.Random(0)


class CatchAdapter:
    def __init__(self, brain, catch_obs_size=4, catch_action_size=3, rng=None):
        self.brain = brain
        self.catch_obs_size = catch_obs_size
        self.catch_action_size = catch_action_size
        self.rng = rng or np.random.default_rng()
        self.W_in = self.rng.normal(0, 0.30, (brain.input_size, catch_obs_size))
        self.b_in = np.zeros(brain.input_size)
        self.W_out = self.rng.normal(0, 0.30, (catch_action_size, brain.output_size))
        self.b_out = np.zeros(catch_action_size)
        # Cache of last brain forward output, used to compute the brain's
        # "intended" mg action for adaptive learning.
        self.last_brain_outputs = None

    def reset(self):
        # Per-episode (one "life"), not per-ball. Hidden persists across balls.
        self.brain.reset()
        self.last_brain_outputs = None

    def policy_logits(self, catch_obs):
        brain_in = self.W_in @ catch_obs + self.b_in
        brain_out = self.brain.forward(brain_in)
        self.last_brain_outputs = brain_out
        return self.W_out @ brain_out + self.b_out

    def _maybe_replay(self, action):
        # Catch's stay (action=1) is the closest analog to mg's `rest`.
        if action == 1 and self.brain._has_episodic():
            self.brain.replay_episode(_REPLAY_RNG)

    def act_greedy(self, catch_obs):
        action = int(np.argmax(self.policy_logits(catch_obs)))
        self._maybe_replay(action)
        return action

    def act_stochastic(self, catch_obs, temperature=1.0):
        logits = self.policy_logits(catch_obs) / max(1e-6, temperature)
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        action = int(self.rng.choice(len(probs), p=probs))
        self._maybe_replay(action)
        return action

    def observe_outcome(self, reward, adaptive_lr=0.0):
        """If adaptive_lr > 0 and this tick produced reward, drive the brain's
        plasticity rule with `reward` as valence on the brain's *intended*
        mg-action (argmax over its 15-dim output, before W_out projection).
        Adaptive-transfer mode (D)."""
        if adaptive_lr <= 0.0 or reward == 0.0 or self.last_brain_outputs is None:
            return
        mg_action = int(np.argmax(self.last_brain_outputs))
        self.brain.learn(mg_action, valence=reward, learning_rate=adaptive_lr)

    def get_params(self):
        return self.W_in.copy(), self.W_out.copy(), self.b_in.copy(), self.b_out.copy()

    def set_params(self, W_in, W_out, b_in, b_out):
        self.W_in = W_in
        self.W_out = W_out
        self.b_in = b_in
        self.b_out = b_out

## Training (Evolution Strategies)

We can't backprop through the brain in pure numpy without a deep-learning library, and we don't want to — the test is purest if the brain is genuinely frozen with no gradient flow. Instead we use Evolution Strategies (ES): perturb the adapter weights with small gaussian noise, evaluate, and update toward perturbations that improved performance. Slow but framework-free, and keeps the brain truly frozen.

In [ ]:
def run_episode(adapter, env, greedy=False, adaptive_lr=0.0):
    adapter.reset()
    obs = env.reset()
    total = 0.0
    done = False
    while not done:
        action = adapter.act_greedy(obs) if greedy else adapter.act_stochastic(obs, temperature=1.0)
        obs, reward, done = env.step(action)
        total += reward
        # Adaptive-transfer mode: feed reward back into brain's plasticity.
        # No-op when adaptive_lr=0 (frozen mode) or for adapters without
        # observe_outcome (e.g., DirectPolicy stub below).
        if hasattr(adapter, 'observe_outcome'):
            adapter.observe_outcome(reward, adaptive_lr=adaptive_lr)
    return total

def evaluate(adapter, env, n_episodes=200, adaptive_lr=0.0):
    return float(np.mean([run_episode(adapter, env, greedy=True, adaptive_lr=adaptive_lr) for _ in range(n_episodes)]))

def train_es(adapter, env, generations=150, pop=32, sigma=0.20, lr=0.20,
             n_avg=3, adaptive_lr=0.0, log_every=10, seed=0, verbose=True):
    """Evolution Strategies. n_avg episodes are averaged per perturbation
    to reduce variance from Catch's stochastic ball positions.
    
    adaptive_lr > 0 enables adaptive-transfer mode: brain's plasticity rules
    fire during episodes (reward feedback drives weights_in/weights_out updates).
    With adaptive_lr=0 the brain is genuinely frozen — only the adapter trains."""
    base = adapter.get_params()
    history = []
    rng = np.random.default_rng(seed)
    for gen in range(generations):
        perturbations = []
        rewards = []
        for _ in range(pop):
            eps = tuple(rng.normal(0, sigma, p.shape) for p in base)
            adapter.set_params(*[b + e for b, e in zip(base, eps)])
            r = float(np.mean([run_episode(adapter, env, greedy=False, adaptive_lr=adaptive_lr) for _ in range(n_avg)]))
            perturbations.append(eps)
            rewards.append(r)
        rewards = np.array(rewards)
        if rewards.std() > 1e-8:
            normed = (rewards - rewards.mean()) / rewards.std()
        else:
            normed = rewards - rewards.mean()
        deltas = [np.zeros_like(p) for p in base]
        for r, eps in zip(normed, perturbations):
            for i in range(len(deltas)):
                deltas[i] += r * eps[i]
        base = tuple(b + (lr / (pop * sigma)) * d for b, d in zip(base, deltas))
        adapter.set_params(*base)
        if gen % log_every == 0 or gen == generations - 1:
            avg = evaluate(adapter, env, n_episodes=20, adaptive_lr=adaptive_lr)
            history.append((gen, float(avg)))
            if verbose:
                print(f'  gen {gen:3d}: eval reward {avg:+.3f}')
    return adapter, history

## The experiment

In [ ]:
# Default sample: v2 brain trained with multi-world selection (resource-preserving).
# 8-hidden-unit brain with 3 episodic memory slots, attention head present.
# Try `learner_champion_hidden14.json` for a v1 brain for comparison.
SAMPLE_BRAIN = os.path.join(BRAIN_REPO, 'transfer/sample_brains/v2_multiworld_overall_champion.json')

# Multi-ball Catch episodes (paddle persists across balls; brain hidden state
# persists too, matching the brain's native operating horizon in mg).
BALLS_PER_EPISODE = 30
ES_GENERATIONS = 100

def random_baseline(env, n_episodes=200, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []
    for _ in range(n_episodes):
        env.reset()
        total = 0.0
        done = False
        while not done:
            action = int(rng.integers(0, 3))
            _, r, done = env.step(action)
            total += r
        rewards.append(total)
    return float(np.mean(rewards))

env = Catch(balls_per_episode=BALLS_PER_EPISODE, rng=np.random.default_rng(7))

print('=== 1) Random policy ===')
random_score = random_baseline(env, n_episodes=200)
print(f'Random policy: {random_score:+.3f}\n')

print('=== 2) Untrained brain + trained adapter (control) ===')
random_brain = TinyBrainInference.random(input_size=72, hidden_size=8, output_size=15,
                                          rng=np.random.default_rng(7),
                                          with_attention=True, with_episodic=True, episodic_capacity=3)
control_adapter = CatchAdapter(random_brain, rng=np.random.default_rng(11))
control_adapter, control_history = train_es(control_adapter, env, generations=ES_GENERATIONS, pop=32, seed=0)
control_score = evaluate(control_adapter, env, n_episodes=200)
print(f'Untrained brain + adapter: {control_score:+.3f}\n')

print('=== 3) Trained brain + trained adapter (transfer test) ===')
print(f'Loading brain from {SAMPLE_BRAIN}')
trained_brain = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
print(f'  hidden={trained_brain.hidden_size}, input={trained_brain.input_size}, output={trained_brain.output_size}')
print(f'  attention head: {trained_brain.attention_weights is not None}')
print(f'  episodic memory: {trained_brain._has_episodic()} '
      f'({"capacity " + str(trained_brain.episodic_slots.shape[0]) if trained_brain._has_episodic() else "none"})\n')
trained_adapter = CatchAdapter(trained_brain, rng=np.random.default_rng(11))
trained_adapter, trained_history = train_es(trained_adapter, env, generations=ES_GENERATIONS, pop=32, seed=0)
trained_score = evaluate(trained_adapter, env, n_episodes=200)
print(f'Trained brain + adapter:   {trained_score:+.3f}')

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if control_history:
    gens, scores = zip(*control_history)
    ax.plot(gens, scores, label='untrained brain + adapter (control)', marker='o')
if trained_history:
    gens, scores = zip(*trained_history)
    ax.plot(gens, scores, label='trained brain + adapter (transfer)', marker='o')
ax.axhline(random_score, color='gray', linestyle='--', label=f'random ({random_score:+.2f})')
ax.set_xlabel('Generation')
ax.set_ylabel('Mean reward (eval)')
ax.set_title('Microcosmic God brain → Catch transfer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nFinal scores (mean reward over 200 episodes, greedy policy):')
print(f'  Random policy:             {random_score:+.3f}')
print(f'  Untrained brain (control): {control_score:+.3f}')
print(f'  Trained brain (transfer):  {trained_score:+.3f}')
print(f'\n  Transfer advantage (trained − control): {trained_score - control_score:+.3f}')

## Skeptic's controls + frozen vs. adaptive

The original transfer test was confounded — single seed, hidden state reset every 9 ticks, brain forced to operate well outside its native temporal horizon. Below: four conditions under both **frozen** (brain weights truly fixed) and **adaptive** (brain's plasticity rules fire from Catch reward signals) modes.

**Four conditions (skeptic controls):**
1. **No-brain direct policy** — Catch is solvable by `action = 1 + sign(ball_x − paddle_x)`, a 12-parameter linear policy. If a direct linear adapter (no brain at all) does as well as the brain-mediated version, the brain isn't contributing.
2. **Permutation control** — take the trained brain, randomly shuffle each weight matrix. Preserves marginal distribution but destroys learned structure. If shuffled transfers as well as unshuffled, the result is "trained brains have better-conditioned statistics" — not "trained brains learned useful representations."
3. **Random-init brain** — brand-new random brain with same architecture (including episodic capacity if v2). Tests whether *training in mg* contributes vs. just having an architecturally-correct brain.
4. **Trained brain** — the actual transfer test.

**Two modes:**
- **Frozen mode (`adaptive_lr=0`):** brain is genuinely fixed during the test. Pure measure of "what was learned in mg."
- **Adaptive mode (`adaptive_lr>0`):** brain's plasticity rules fire from per-ball reward signals. Tests "can this brain rapidly *adapt* to a new task using its native learning machinery?" — the brain's natural mode of operation.

**The decisive contrasts:**
- `trained − permuted` (frozen): structure matters?
- `trained − random_brain` (frozen): mg training adds something beyond architecture?
- `trained − direct` (frozen): brain helps vs. no brain?
- `trained_adaptive − trained_frozen`: does the brain's plasticity machinery actually help on Catch?

In [ ]:
def shuffle_brain(brain, rng):
    """Permute each weight matrix independently. Preserves marginal
    distribution of weights while destroying learned input-output and
    input-hidden structure. Also shuffles episodic slots so the v2
    permuted control isn't quietly retaining its episodic memories."""
    def _shuf(arr):
        if arr is None:
            return None
        flat = arr.flatten().copy()
        rng.shuffle(flat)
        return flat.reshape(arr.shape)
    return TinyBrainInference(
        weights_in=_shuf(brain.weights_in),
        weights_out=_shuf(brain.weights_out),
        bias_h=_shuf(brain.bias_h),
        bias_o=_shuf(brain.bias_o),
        attention_weights=_shuf(brain.attention_weights),
        attention_bias=_shuf(brain.attention_bias),
        episodic_slots=_shuf(brain.episodic_slots),
        episodic_age=_shuf(brain.episodic_age),
    )


class DirectPolicy:
    """4 -> 3 linear policy with no brain at all. Drop-in replacement for
    CatchAdapter: same interface, ES can train it the same way. Tests
    whether the brain is helping or just being routed-around by the adapter.
    No `observe_outcome` -> adaptive_lr has no effect (no brain to learn)."""

    def __init__(self, catch_obs_size=4, catch_action_size=3, rng=None):
        self.rng = rng or np.random.default_rng()
        self.W = self.rng.normal(0, 0.30, (catch_action_size, catch_obs_size))
        self.b = np.zeros(catch_action_size)

    def reset(self):
        pass

    def policy_logits(self, obs):
        return self.W @ obs + self.b

    def act_greedy(self, obs):
        return int(np.argmax(self.policy_logits(obs)))

    def act_stochastic(self, obs, temperature=1.0):
        l = self.policy_logits(obs) / max(1e-6, temperature)
        p = np.exp(l - l.max())
        p /= p.sum()
        return int(self.rng.choice(len(p), p=p))

    def get_params(self):
        return (self.W.copy(), self.b.copy())

    def set_params(self, W, b):
        self.W = W
        self.b = b

## Run all four conditions × two modes × N seeds

5 seeds × 4 conditions × 2 modes (frozen, adaptive) × ES generations.

Each "episode" is `BALLS_PER_EPISODE` consecutive balls — the brain's hidden state persists across balls within an episode (matches its native temporal horizon in mg). The adapter resets per-episode (one "life") not per-ball.

Defaults are tuned to run in ~1-2 hours on Colab CPU. Reduce `N_SEEDS` or `ES_GENERATIONS_SKEPTIC` for a faster sanity check, raise for tighter estimates.

In [ ]:
def run_seed(make_adapter, seed, gens=80, pop=24, n_avg=3, eval_eps=200,
             balls_per_episode=BALLS_PER_EPISODE, adaptive_lr=0.0):
    env = Catch(balls_per_episode=balls_per_episode, rng=np.random.default_rng(seed))
    adapter = make_adapter(seed)
    adapter, _ = train_es(
        adapter, env,
        generations=gens, pop=pop, n_avg=n_avg,
        adaptive_lr=adaptive_lr,
        seed=seed, log_every=gens, verbose=False,
    )
    return evaluate(adapter, env, n_episodes=eval_eps, adaptive_lr=adaptive_lr)


# ----------- Tunable parameters -----------
N_SEEDS = 5
ES_GENERATIONS_SKEPTIC = 80
ADAPTIVE_LR = 0.10  # adaptive-mode plasticity strength
SEEDS = list(range(20, 20 + N_SEEDS))
# ------------------------------------------

trained_brain = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
print(f'Loaded brain: hidden={trained_brain.hidden_size}, '
      f'episodic={trained_brain._has_episodic()} '
      f'(capacity={trained_brain.episodic_slots.shape[0] if trained_brain._has_episodic() else 0})\n')

results = {}

for mode_label, mode_lr in [('frozen', 0.0), ('adaptive', ADAPTIVE_LR)]:
    print(f'================  MODE: {mode_label.upper()} (adaptive_lr={mode_lr})  ================\n')

    print('A) Direct linear policy (no brain) ...')
    results[f'direct_{mode_label}'] = [
        run_seed(lambda s: DirectPolicy(rng=np.random.default_rng(s)), s,
                 gens=ES_GENERATIONS_SKEPTIC, adaptive_lr=mode_lr)
        for s in SEEDS
    ]
    print('   ' + ' '.join(f'{x:+.2f}' for x in results[f'direct_{mode_label}']))

    print('B) Random-init brain + adapter ...')
    cap = trained_brain.episodic_slots.shape[0] if trained_brain._has_episodic() else 0
    def random_brain_factory(s, _cap=cap):
        rb = TinyBrainInference.random(input_size=72, hidden_size=trained_brain.hidden_size, output_size=15,
                                        rng=np.random.default_rng(s + 100),
                                        with_attention=True, with_episodic=(_cap > 0), episodic_capacity=_cap)
        return CatchAdapter(rb, rng=np.random.default_rng(s + 1000))
    results[f'random_brain_{mode_label}'] = [
        run_seed(random_brain_factory, s, gens=ES_GENERATIONS_SKEPTIC, adaptive_lr=mode_lr)
        for s in SEEDS
    ]
    print('   ' + ' '.join(f'{x:+.2f}' for x in results[f'random_brain_{mode_label}']))

    print('C) Permuted trained brain + adapter ...')
    def permuted_brain_factory(s):
        pb = shuffle_brain(trained_brain, rng=np.random.default_rng(s + 2000))
        return CatchAdapter(pb, rng=np.random.default_rng(s + 1000))
    results[f'permuted_{mode_label}'] = [
        run_seed(permuted_brain_factory, s, gens=ES_GENERATIONS_SKEPTIC, adaptive_lr=mode_lr)
        for s in SEEDS
    ]
    print('   ' + ' '.join(f'{x:+.2f}' for x in results[f'permuted_{mode_label}']))

    print('D) Trained brain + adapter (transfer test) ...')
    def trained_brain_factory(s):
        # Fresh deep copy via from_checkpoint so adaptive learning doesn't
        # corrupt the shared trained_brain across seeds.
        tb = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
        return CatchAdapter(tb, rng=np.random.default_rng(s + 1000))
    results[f'trained_{mode_label}'] = [
        run_seed(trained_brain_factory, s, gens=ES_GENERATIONS_SKEPTIC, adaptive_lr=mode_lr)
        for s in SEEDS
    ]
    print('   ' + ' '.join(f'{x:+.2f}' for x in results[f'trained_{mode_label}']))
    print()

## The decisive comparison

Box plot across seeds, plus the three key contrasts.

In [ ]:
labels = ['Direct\n(no brain)', 'Random-init\nbrain', 'Permuted\ntrained brain', 'Trained\nbrain']
condition_keys = ['direct', 'random_brain', 'permuted', 'trained']

frozen_data = [results[f'{k}_frozen'] for k in condition_keys]
adaptive_data = [results[f'{k}_adaptive'] for k in condition_keys]

fig, ax = plt.subplots(figsize=(12, 6))
positions = np.arange(len(condition_keys))
width = 0.35
bp_frozen = ax.boxplot(frozen_data, positions=positions - width/2, widths=width, patch_artist=True)
bp_adaptive = ax.boxplot(adaptive_data, positions=positions + width/2, widths=width, patch_artist=True)
for patch in bp_frozen['boxes']:
    patch.set_facecolor('lightblue')
for patch in bp_adaptive['boxes']:
    patch.set_facecolor('lightcoral')
jitter_rng = np.random.default_rng(99)
for i, scores in enumerate(frozen_data):
    x = (positions[i] - width/2) + jitter_rng.uniform(-0.06, 0.06, size=len(scores))
    ax.scatter(x, scores, color='black', s=20, zorder=3)
for i, scores in enumerate(adaptive_data):
    x = (positions[i] + width/2) + jitter_rng.uniform(-0.06, 0.06, size=len(scores))
    ax.scatter(x, scores, color='black', s=20, zorder=3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean reward (eval, 200 episodes per seed)')
ax.set_title(f'Catch transfer test - {N_SEEDS} seeds, multi-ball ({BALLS_PER_EPISODE}/episode), frozen vs adaptive')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='lightblue', label='frozen brain'),
                   Patch(facecolor='lightcoral', label='adaptive (brain plasticity on)')],
          loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nResults table (mean +/- std, n={N_SEEDS} seeds, 200 eval episodes each):')
print(f'  {"Condition":24s}  {"Frozen":>20s}  {"Adaptive":>20s}')
for k, lbl in zip(condition_keys, ['Direct (no brain)', 'Random-init brain', 'Permuted trained brain', 'Trained brain']):
    f_arr = np.array(results[f'{k}_frozen'])
    a_arr = np.array(results[f'{k}_adaptive'])
    print(f'  {lbl:24s}  {f_arr.mean():+.3f} +/- {f_arr.std():.3f}      {a_arr.mean():+.3f} +/- {a_arr.std():.3f}')

t_frozen = np.mean(results['trained_frozen'])
t_adaptive = np.mean(results['trained_adaptive'])
print('\nKey contrasts (frozen mode):')
print(f'  trained - random_brain: {t_frozen - np.mean(results["random_brain_frozen"]):+.3f}')
print(f'  trained - permuted:     {t_frozen - np.mean(results["permuted_frozen"]):+.3f}')
print(f'  trained - direct:       {t_frozen - np.mean(results["direct_frozen"]):+.3f}')
print('\nAdaptive vs frozen (does plasticity machinery help on Catch?):')
print(f'  trained:      {t_adaptive - t_frozen:+.3f}')
print(f'  random_brain: {np.mean(results["random_brain_adaptive"]) - np.mean(results["random_brain_frozen"]):+.3f}')
print(f'  permuted:     {np.mean(results["permuted_adaptive"]) - np.mean(results["permuted_frozen"]):+.3f}')

## Interpreting the result

**Reward range:** Catch returns +1 on a catch, −1 on a miss. Per-episode reward is in {−1, +1}. A perfect player scores +1.0; uniform-random scores around 0.0.

**Reading the contrasts:**

| Contrast | What it tells you |
|---|---|
| `trained − permuted` | Whether *learned structure* (vs. just weight statistics) does work. If positive, shuffling the weights destroyed something useful. |
| `trained − random_brain` | Whether the alife training specifically produced something that transfers, beyond what any random brain would do. |
| `trained − direct` | Whether the brain helps at all vs. solving Catch with a 12-parameter linear policy. |

### What we observed (10 seeds, 150 ES generations, learner_champion_hidden14)

```
Direct (no brain)     : +0.084 ± 0.252
Random-init brain     : +0.004 ± 0.400
Permuted trained brain: -0.227 ± 0.197
Trained brain         : -0.020 ± 0.360

trained − permuted     = +0.207   ✓ structure matters
trained − random_brain = -0.024   ✗ no transfer beyond random
trained − direct       = -0.104   ✗ brain hinders slightly
```

**The permutation test passes** — shuffling weights makes the brain meaningfully worse. So the substrate IS producing structured representations, not just brains with reasonable weight statistics. That's a real positive finding.

**But the structure doesn't help with Catch specifically.** Trained and random brains transfer equally well; a direct linear policy slightly beats both. This is consistent with the read that Catch is too simple a test target — it's solvable by `action = 1 + sign(ball_x − paddle_x)`, a 12-parameter linear policy. A brain with rich representations of mg's world gets in the way of the adapter trying to extract that simple rule.

### What this means for the transferable-minds claim

The substrate is producing *structured cognition*, not just well-tuned random functions. That's necessary for transfer but isn't sufficient: structure has to be *relevant* to the target task.

A better future transfer target would be a task that genuinely requires:
- **temporal reasoning** (memory, prediction across multiple steps)
- **causal sequencing** (multi-step actions where order matters)
- **partial observability** (the brain's hidden state has to track unseen information)

The mg substrate was selected for exactly these. Catch tests none of them.

Candidates for a stronger second transfer test: a memory-based maze, a multi-step puzzle box (e.g., open-this-then-pull-that), or even a sequential prediction task where the brain's recurrent dynamics actually matter.

For now: the permutation result is the meaningful signal. The Catch result tells you Catch isn't the right probe.